# SQL Part 3: Sorting and Aggregating Data

## Learning Objectives

By the end of this notebook, you will be able to:
- Sort query results using `ORDER BY` with one or more columns
- Summarise data using aggregate functions: `COUNT`, `MIN`, `MAX`, `SUM`, `AVG`
- Group results by a column using `GROUP BY`
- Filter grouped results using `HAVING`
- Use `AS`, `ROUND`, and `SUBSTRING` to shape your output

---

## Context Recap

This notebook builds on the filtering skills from [Notebook 2](2_filtering_data.ipynb) — you will combine `WHERE` with `ORDER BY` and use `GROUP BY` to summarise data in ways that answer real analytical questions.

This is the longest notebook in the series. Take your time, and feel free to take a short break between sections.

**Prerequisites:** Complete [2 - Filtering Data](2_filtering_data.ipynb) before starting.

Run the setup cell below before starting. If your kernel restarts at any point, run it again.

In [ ]:
# Setup: load the SQL extension and connect to the pagila database
%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = False
%config SqlMagic.autopolars = False
%config SqlMagic.displaycon = False

%sql sqlite:///data/pagila.db

---

## Section 3.1: Sorting data with `ORDER BY`

By default, SQL returns records in no guaranteed order — the database engine returns them however it pleases. Use `ORDER BY` to control the sort:

```sql
SELECT *
FROM table_name
ORDER BY column_name;        -- ascending (A→Z, 0→9) — this is the default
ORDER BY column_name DESC;   -- descending (Z→A, 9→0)
ORDER BY col1, col2;         -- sort by col1 first, then by col2 to break ties
```

`ORDER BY` is often combined with `LIMIT` to answer questions like "What is the largest?" or "Which comes first?" — you'll use this pattern several times in this section.

##### Question 43

<blockquote>

Sort customers alphabetically by `last_name`. Which name is first? Which is last?

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 43</summary>

```sql
%%sql
SELECT first_name, last_name
FROM customer
ORDER BY last_name;
```

`Rafael Abney` is first; `Cynthia Young` is last. By default, `ORDER BY` sorts ascending — A before Z.

</details>

##### Question 44

<blockquote>

Sort the records in `payment` from the largest `amount` to the smallest.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 44</summary>

```sql
%%sql
SELECT *
FROM payment
ORDER BY amount DESC;
```

`DESC` reverses the default order. The largest payments appear at the top — useful when you want to identify high-value transactions quickly.

</details>

##### Question 45

<blockquote>

Return only 1 record — the customer whose `first_name` comes first alphabetically.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 45</summary>

```sql
%%sql
SELECT *
FROM customer
ORDER BY first_name
LIMIT 1;
```

`Aaron Selby` is first alphabetically. Combining `ORDER BY` with `LIMIT 1` is a useful pattern for finding the single highest or lowest value in any column — you'll use it often.

</details>

##### Question 46

<blockquote>

What are the first and last cities alphabetically?

<details>
<summary>&#128161; Hint</summary>

You will need two queries.

</details>

</blockquote>

In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution to Question 46</summary>

```sql
%%sql
SELECT city FROM city ORDER BY city LIMIT 1;
```
`A Corua (La Corua)` is first.

```sql
%%sql
SELECT city FROM city ORDER BY city DESC LIMIT 1;
```
`s-Hertogenbosch` is last. Note that lowercase letters sort after uppercase in SQLite, which is why `s-Hertogenbosch` ends up at the bottom rather than among the S entries.

</details>

##### Question 47

<blockquote>

Which countries come first and last alphabetically?

</blockquote>

In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution to Question 47</summary>

```sql
%%sql
SELECT country FROM country ORDER BY country LIMIT 1;
```
Afghanistan is first.

```sql
%%sql
SELECT country FROM country ORDER BY country DESC LIMIT 1;
```
Zambia is last.

</details>

##### Question 48

<blockquote>

Extract the top 15 films ordered by `length` descending. Only show `film_id`, `title`, and `length`.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 48</summary>

```sql
%%sql
SELECT film_id, title, length
FROM film
ORDER BY length DESC
LIMIT 15;
```

Selecting only the columns you need (rather than `SELECT *`) makes long results easier to read and is good practice when working with real databases that may have dozens of columns.

</details>

##### Question 49

<blockquote>

Sort films by `length` descending, and break ties alphabetically by `title`.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 49</summary>

```sql
%%sql
SELECT film_id, title, length
FROM film
ORDER BY length DESC, title;
```

When two films share the same length, the second sort column (`title`) determines their order. This two-column sort pattern is useful whenever you want a consistent, predictable ordering.

</details>

##### Question 50

<blockquote>

- Which is the biggest `postal_code` in the `address` table?
- Which is the smallest?
- `postal_code` is stored as text, not a number — what problems does this cause when sorting?

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 50</summary>

```sql
%%sql
SELECT postal_code FROM address ORDER BY postal_code DESC;
```

The biggest appears to be 99865; the smallest 1027 (ignoring blanks).

This is a common data quality issue. When numeric data is stored as text, sorting is alphabetical rather than numerical — character by character from left to right. This means `'9'` sorts after `'10'` because `'9' > '1'`, even though 9 is less than 10 as a number. You would see `99865` at the top, but `1027` near the bottom rather than the true minimum. This kind of storage choice leads to misleading sorts and can cause bugs in downstream analysis — always check the data type of a column before sorting it.

</details>

##### Question 51

<blockquote>

Which is the longest film with a `PG` rating?

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 51</summary>

```sql
%%sql
SELECT title, length, rating
FROM film
WHERE rating = 'PG'
ORDER BY length DESC
LIMIT 1;
```

`WORST BANGER` at 185 minutes. This combines `WHERE` (from Notebook 2) with `ORDER BY` and `LIMIT` — you will use this combination regularly.

</details>

##### Question 52

<blockquote>

What is the largest payment made by a customer whose `first_name` begins with `Y`?

<details>
<summary>&#128161; Hint</summary>

You will need multiple queries — first find the customer IDs, then find the largest payment.

</details>

</blockquote>

In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution to Question 52</summary>

```sql
%%sql
SELECT customer_id, first_name
FROM customer
WHERE first_name LIKE 'Y%';
```
Customer IDs 174 and 190.

```sql
%%sql
SELECT amount
FROM payment
WHERE customer_id IN (174, 190)
ORDER BY amount DESC
LIMIT 1;
```
Largest payment is 9.99. Notice how this query chains together `LIKE`, `IN`, `ORDER BY`, and `LIMIT` — each concept you have learned so far is now a tool you can combine with others.

</details>

---

## Section 3.2: Aggregate functions and `GROUP BY`

Sorting lets you reorder data. **Aggregate functions** do something more powerful: they collapse multiple rows into a single summary value.

| Function | Returns |
|---|---|
| `COUNT(*)` | Number of rows |
| `MIN(col)` | Smallest value |
| `MAX(col)` | Largest value |
| `SUM(col)` | Total of all values |
| `AVG(col)` | Mean (average) |

On their own, these functions summarise the entire table. Use `GROUP BY` to apply them separately to each group of rows:

```sql
SELECT store_id,
       COUNT(*) AS num_customers
FROM customer
GROUP BY store_id;
```

The `AS` keyword creates an **alias** — a label for a computed column in the output. Without an alias, aggregate columns appear with unhelpful names like `COUNT(*)`.

You can also round numeric output with `ROUND(value, decimal_places)`:

```sql
SELECT ROUND(AVG(length), 2) AS avg_length
FROM film;
```

Run the worked example below to see `GROUP BY` in action:

In [ ]:
%%sql
SELECT store_id,
       COUNT(*) AS num_customers
FROM customer
GROUP BY store_id;

This query groups all customers by their store and counts how many belong to each group. Store 1 has 326 customers; store 2 has 273. The `AS` keyword gives the count column a meaningful name — without it you would see `COUNT(*)` in the header, which is harder to read in a report or a shared dataset.

##### Question 53

<blockquote>

What is the minimum, maximum, sum, and mean of `amount` in the `payment` table?

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 53</summary>

```sql
%%sql
SELECT COUNT(amount),
       MIN(amount),
       MAX(amount),
       SUM(amount),
       AVG(amount)
FROM payment;
```

- Count: 16,049
- Min: 0.00
- Max: 11.99
- Sum: 67,416.51
- Average: 4.20

A minimum of 0.00 is worth noting — some rentals generated no payment. This kind of anomaly is easy to spot with aggregate functions and would be worth investigating in a real dataset.

</details>

##### Question 54

<blockquote>

Use `MIN` and `MAX` to find the first and last film titles alphabetically.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 54</summary>

```sql
%%sql
SELECT MIN(title), MAX(title)
FROM film;
```

First: `ACADEMY DINOSAUR`. Last: `ZORRO ARK`. This demonstrates that `MIN` and `MAX` work on text columns as well as numeric ones — they apply alphabetical ordering to find the first and last values.

</details>

##### Question 55

<blockquote>

Create a single query showing the minimum, maximum, sum, and mean of `amount` in `payment`. Use aliases to give each column a meaningful name in the output.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 55</summary>

```sql
%%sql
SELECT
    COUNT(amount) AS num_records,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    SUM(amount) AS total_amount,
    AVG(amount) AS avg_amount
FROM payment;
```

The results are the same as Question 53, but with aliases the output is ready to share — column names like `min_amount` and `total_amount` are self-explanatory to anyone reading the results.

</details>

##### Question 56

<blockquote>

What is the minimum, maximum, sum, and mean of `rental_duration` in `film`?

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 56</summary>

```sql
%%sql
SELECT
    MIN(rental_duration),
    MAX(rental_duration),
    SUM(rental_duration),
    AVG(rental_duration)
FROM film;
```

Min: 3, Max: 7, Sum: 4,985, Average: 4.99. Every film in this database has a rental period between 3 and 7 days.

</details>

##### Question 57

<blockquote>

Choose another numeric column from `film`. Create a summary with min, max, sum, and mean. Use `ROUND()` to format the mean to 2 decimal places.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Example Solution — Question 57</summary>

```sql
%%sql
SELECT
    MIN(length) AS min_length,
    MAX(length) AS max_length,
    SUM(length) AS total_length,
    ROUND(AVG(length), 2) AS avg_length
FROM film;
```

`ROUND(AVG(length), 2)` formats the mean to 2 decimal places — without it, SQLite returns many decimal places that add noise rather than precision.

</details>

##### Question 58

<blockquote>

Do `PG` films or `PG-13` films tend to be longer on average?

</blockquote>

In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution to Question 58</summary>

```sql
%%sql
SELECT AVG(length)
FROM film
WHERE rating = 'PG';
```
PG average: 112.01 minutes.

```sql
%%sql
SELECT AVG(length)
FROM film
WHERE rating = 'PG-13';
```
PG-13 average: 120.44 minutes — PG-13 films tend to be longer. This approach uses two separate queries with `WHERE` to compare groups. In the next few questions, you will see how `GROUP BY` handles this in a single query.

</details>

##### Question 59

<blockquote>

Aggregate the `customer` table by `store_id`. Create a column with the number of customers for each store.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 59</summary>

```sql
%%sql
SELECT
    store_id,
    COUNT(*) AS num_customers
FROM customer
GROUP BY store_id;
```

This is the same result as the worked example at the top of this section — now try writing it from scratch.

</details>

##### Question 60

<blockquote>

Create a summary showing the average `length` of films for each distinct `rating`.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 60</summary>

```sql
%%sql
SELECT
    rating,
    AVG(length) AS avg_length
FROM film
GROUP BY rating;
```

This answers Question 58 in a single query — every rating appears as one row, and you can compare them all at a glance. This is one of the main advantages of `GROUP BY` over running separate `WHERE` queries.

</details>

##### Question 61

<blockquote>

Aggregate the `film` table by `release_year` and `rating`. Show the count of films and the first film alphabetically for each combination.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 61</summary>

```sql
%%sql
SELECT
    release_year,
    rating,
    COUNT(*) AS frequency,
    MIN(title) AS first_film_alphabetically
FROM film
GROUP BY release_year, rating;
```

When you `GROUP BY` two columns, each unique combination of those columns becomes its own row. You can group by as many columns as you need.

</details>

##### Question 62

<blockquote>

Aggregate `film` by `rating`. Show the count of films for each rating.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 62</summary>

```sql
%%sql
SELECT
    rating,
    COUNT(film_id) AS num_records
FROM film
GROUP BY rating;
```

`COUNT(film_id)` counts non-null values in the `film_id` column. For this table that gives the same result as `COUNT(*)`, but `COUNT(column_name)` is more precise when a column might contain NULLs.

</details>

##### Question 63

<blockquote>

Summarise the average sales value for each member of staff in the `payment` table.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 63</summary>

```sql
%%sql
SELECT
    staff_id,
    AVG(amount) AS avg_sales_value
FROM payment
GROUP BY staff_id;
```

Staff ID 1 averages 4.17 per transaction; staff ID 2 averages 4.23. The difference is small — but in a real retail or service context, tracking average transaction value by staff member is a standard performance metric.

</details>

##### Question 64

<blockquote>

Aggregate `payment` by `staff_id` and `amount`. Show the count of records for each combination.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 64</summary>

```sql
%%sql
SELECT
    staff_id,
    amount,
    COUNT(*) AS frequency
FROM payment
GROUP BY staff_id, amount;
```

This reveals how often each staff member processed each specific payment amount. You can see patterns in the pricing structure — certain amounts appear far more frequently than others.

</details>

##### Question 65

<blockquote>

Create a table showing how many customers have a `first_name` starting with each letter of the alphabet. Use the `SUBSTRING(string, start, length)` function.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 65</summary>

```sql
%%sql
SELECT
    SUBSTRING(first_name, 1, 1) AS first_letter,
    COUNT(*) AS frequency
FROM customer
GROUP BY first_letter;
```

`SUBSTRING(string, start_position, length)` extracts part of a string. Here it extracts the first character (position 1, length 1), which is then used as a grouping key. This technique — extracting part of a string to group by — is useful whenever you need to summarise data at a coarser level than the original values (for example, grouping by year from a date stored as text, or by department code from an employee ID).

</details>

---

## Section 3.3: Filtering groups with `HAVING`

`GROUP BY` can be combined with `ORDER BY`, `WHERE`, and a keyword you have not seen yet: `HAVING`.

- **`WHERE`** filters rows **before** grouping
- **`HAVING`** filters groups **after** grouping

This distinction matters because aggregate functions like `COUNT(*)` do not exist until after grouping has happened. A common mistake is trying to filter on an aggregate in the `WHERE` clause — for example:

```sql
-- This does NOT work:
SELECT store_id, COUNT(*) AS num_customers
FROM customer
WHERE COUNT(*) > 300
GROUP BY store_id;
```

`WHERE` runs before the grouping, so `COUNT(*)` has not been calculated yet. Use `HAVING` instead:

```sql
-- This works:
SELECT store_id, COUNT(*) AS num_customers
FROM customer
GROUP BY store_id
HAVING COUNT(*) > 300;
```

`HAVING` runs after grouping, so it can filter on aggregated values. Run the example below to see it in action:

In [ ]:
%%sql
SELECT store_id,
       COUNT(*) AS num_customers
FROM customer
GROUP BY store_id
HAVING COUNT(*) > 300;

Only store 1 has more than 300 customers, so only one row is returned. Store 2 (with 273 customers) is filtered out. You can use any aggregate function in a `HAVING` clause — `HAVING SUM(amount) > 190`, `HAVING AVG(length) < 100`, and so on.

##### Question 66

<blockquote>

Which `customer_id` has made the most payments? How many?

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 66</summary>

```sql
%%sql
SELECT
    customer_id,
    COUNT(amount) AS num_payments
FROM payment
GROUP BY customer_id
ORDER BY num_payments DESC
LIMIT 1;
```

Customer ID 148 with 46 transactions. This uses `GROUP BY` with `ORDER BY` and `LIMIT` — a combination that answers "who has the most?" questions efficiently.

</details>

##### Question 67

<blockquote>

How much did that customer spend in total?

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 67</summary>

```sql
%%sql
SELECT
    customer_id,
    COUNT(amount) AS num_payments,
    SUM(amount) AS total_payment
FROM payment
GROUP BY customer_id
ORDER BY num_payments DESC;
```

Customer ID 148 spent a total of 216.54. You can include multiple aggregate functions in the same `SELECT` — here, counting transactions and summing their amounts in one query.

</details>

##### Question 68

<blockquote>

List all `customer_id` values where the customer has spent more than 190 in total.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 68</summary>

```sql
%%sql
SELECT
    customer_id,
    SUM(amount) AS total_payment
FROM payment
GROUP BY customer_id
HAVING SUM(amount) > 190;
```

This is `HAVING` in action: the filter applies to `SUM(amount)`, which only exists after grouping. You cannot write `WHERE SUM(amount) > 190` here — `HAVING` is the correct tool.

</details>

##### Question 69

<blockquote>

List each `customer_id` alongside their maximum single payment. Order by that maximum payment descending.

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 69</summary>

```sql
%%sql
SELECT
    customer_id,
    MAX(amount) AS max_payment
FROM payment
GROUP BY customer_id
ORDER BY MAX(amount) DESC;
```

You can use aggregate functions directly in `ORDER BY` without repeating the alias. This shows which customers have ever made high-value single payments — a different insight from total spend.

</details>

##### Question 70

<blockquote>

Which `customer_id` has the highest average transaction amount?

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 70</summary>

```sql
%%sql
SELECT
    customer_id,
    AVG(amount) AS avg_payment
FROM payment
GROUP BY customer_id
ORDER BY avg_payment DESC
LIMIT 1;
```

Customer ID 187 with an average of 5.70. Notice that this is not the same customer as the one with the most payments (Q66) or the highest total spend (Q67) — average value, frequency, and total spend measure different aspects of customer behaviour.

</details>

##### Question 71

<blockquote>

List the `customer_id` values with an average transaction amount more than `5`. How many are there?

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 71</summary>

```sql
%%sql
SELECT
    customer_id,
    AVG(amount) AS avg_payment
FROM payment
GROUP BY customer_id
HAVING AVG(amount) > 5;
```

23 customers. `HAVING AVG(amount) > 5` keeps only the groups where the calculated average exceeds 5 — groups below that threshold are discarded from the result.

</details>

##### Question 72

<blockquote>

Which film `rating` tends to have the longest films? Which rating has the highest average `rental_rate`?

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 72</summary>

```sql
%%sql
SELECT
    rating,
    AVG(length) AS avg_length,
    AVG(rental_rate) AS avg_rental_rate
FROM film
GROUP BY rating;
```

PG-13 films are longest on average. PG films have the highest average rental rate. Note the distinction: length and rental rate are separate attributes — a rating category can be long without being expensive.

</details>

##### Question 73

<blockquote>

Find all `city_id` values that appear more than once in the `address` table (duplicated cities).

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 73</summary>

```sql
%%sql
SELECT
    city_id,
    COUNT(*) AS freq
FROM address
GROUP BY city_id
HAVING COUNT(*) > 1;
```

Duplicated city IDs: 42, 300, 312, 576. This is a real-world data quality pattern — using `GROUP BY` with `HAVING COUNT(*) > 1` is a standard technique for finding duplicates in any table.

</details>

##### Question 74

<blockquote>

Which customer has rented films at the most different price-points? How many different price-points?

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 74</summary>

```sql
%%sql
SELECT
    customer_id,
    COUNT(DISTINCT amount) AS num_price_points
FROM payment
GROUP BY customer_id
ORDER BY num_price_points DESC;
```

Customer ID 448 with 13 different price-points. `COUNT(DISTINCT amount)` counts unique values within each group — combining `DISTINCT` with aggregation and `GROUP BY` in a single query.

</details>

##### Question 75

<blockquote>

What is the most common starting letter of all film titles? What is the second most common?

<details>
<summary>&#128161; Hint</summary>

Use the `SUBSTRING` function to extract the first letter of each title.

</details>

</blockquote>

In [ ]:
%%sql


<details>
<summary>Solution to Question 75</summary>

```sql
%%sql
SELECT
    SUBSTRING(title, 1, 1) AS film_first_letter,
    COUNT(*) AS freq
FROM film
GROUP BY SUBSTRING(title, 1, 1)
ORDER BY COUNT(*) DESC;
```

Most common: S. Second most common: C. You can use `SUBSTRING` directly in both `SELECT` and `GROUP BY` — you do not need to alias it first.

</details>

##### Question 76

<blockquote>

Which `rental_id` accounts for the most sales? What is the name of the film that rental belongs to? This will require multiple queries.

</blockquote>

In [ ]:
%%sql


In [ ]:
%%sql


In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution to Question 76</summary>

```sql
%%sql
SELECT rental_id, ROUND(SUM(amount), 2) AS total_sales
FROM payment
GROUP BY rental_id
ORDER BY SUM(amount) DESC
LIMIT 1;
```
Rental ID 4591 with 13.94 total.

```sql
%%sql
SELECT rental_id, inventory_id
FROM rental
WHERE rental_id = 4591;
```
Inventory ID 2276.

```sql
%%sql
SELECT inventory_id, film_id
FROM inventory
WHERE inventory_id = 2276;
```
Film ID 492.

```sql
%%sql
SELECT film_id, title FROM film WHERE film_id = 492;
```
The film is `JUNGLE CLOSER`.

This question required you to trace a value across four separate tables — `payment` → `rental` → `inventory` → `film` — using the result of each query as the input to the next. This manual chaining is the motivation for JOINs: in the next notebook, you will learn how to combine all four tables in a single query.

</details>

---

## Summary

In this notebook you have covered:

| Keyword / Function | Purpose |
|---|---|
| `ORDER BY col` | Sort ascending (A→Z, 0→9) |
| `ORDER BY col DESC` | Sort descending (Z→A, 9→0) |
| `ORDER BY col1, col2` | Sort by multiple columns; second column breaks ties |
| `COUNT`, `MIN`, `MAX`, `SUM`, `AVG` | Aggregate functions — collapse many rows into one value |
| `GROUP BY col` | Apply aggregate functions to each group separately |
| `HAVING condition` | Filter groups after aggregation (use instead of `WHERE` for aggregates) |
| `AS alias` | Give a column a custom name in the output |
| `ROUND(val, n)` | Round a number to `n` decimal places |
| `SUBSTRING(str, start, len)` | Extract part of a string |

**Next:** [4 - Joins](4_joins.ipynb)